# Make figure panels from your shape clusters

The first notebook produced a shape map and shape clusters. This one turns them
into **figure panels** you can put in a talk or a paper: every panel is saved as
an **SVG** (vector, text stays editable in Inkscape or Illustrator) and as a
300 dpi **PNG**, in the `figures/` folder.

What you do:

1. Point the notebook at `shape_embedding.csv` (the EDIT cell, section 1). Optionally add a small table of your own labels per stack.
2. Choose the columns you want to colour the map by.
3. Run the cells from top to bottom.

Panels in this notebook:

| Section | Panel |
|---|---|
| 3 | The map coloured by shape cluster |
| 4 | The same map coloured by each column you list in `COLOR_BY` |
| 5 | Bars: which clusters make up each group (any column) |
| 6 | Heatmaps: group by cluster, share and over/under-representation |
| 7 | One dot per stack (only when the table has several stacks) |
| 8 | The map with a numbered badge at the centre of each cluster |
| 9 | A grey "ghost" map with one marker per cluster, to finish in a vector editor |
| 10 | Pictures of typical cells (optional) |

Every panel uses the same type sizes and colours, so panels from different
sections look alike when you put them side by side.

In [ ]:
# Make the helper file next to this notebook importable (works from the repo root or from this folder).
import sys
from pathlib import Path

for _p in (Path.cwd(), Path.cwd() / "tutorials" / "analysis", Path.cwd().parent):
    if (_p / "_helpers.py").exists():
        sys.path.insert(0, str(_p))
        break
else:
    raise FileNotFoundError("Could not find _helpers.py. Open this notebook from the repository root or from tutorials/analysis.")

import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import _helpers as h

warnings.filterwarnings("ignore", category=FutureWarning)
h.apply_style()
print("Ready.")

## 1. Edit this cell

This is the only cell you must change.

- `SHAPE_EMBEDDING_CSV`: the file written by the first notebook.
- `METADATA_CSV`: optional. One row per stack: a `stack_name` column plus any columns you like (sample, region, treatment...). Use `None` if you have none. Columns already in the embedding file are kept as they are.
- `RESOLUTION`: which cluster resolution to show. It must have been computed by the first notebook (a column called `leiden_r<resolution>`).
- `COLOR_BY`: columns to colour the map by, one panel each. Text columns and columns with few distinct values get one colour per value. Other numeric columns get a colour bar.
- `GROUP_COL`: the column that defines the groups for the bar and heatmap sections (5 and 6). `None` uses `stack_name` when the table has several stacks, otherwise the first column of `COLOR_BY`.
- `COLOR_TRANSFORM`: how a numeric `COLOR_BY` column becomes colours (section 4): `"raw"`, `"rank"` (spreads the colours evenly over the cells; good for skewed measurements), `"log1p"`, `"sqrt"`, `"zscore"` or `"winsorize"`.
- `FIGURE_SIZE`: `"half"` (86 x 70 mm, one column of a paper), `"full"` (180 x 130 mm), `"a4_landscape"` or `"a4_portrait"`.
- `LABELS_TIFF`, `MESHES_DIR`: optional, only for the pictures of typical cells (section 10).

In [ ]:
# ---------------- EDIT ME ----------------
SHAPE_EMBEDDING_CSV = "shape_output/shape_embedding.csv"
METADATA_CSV = None            # for example "metadata.csv"
OUTPUT_DIR = "figure_output"

RESOLUTION = 1.0               # cluster resolution to show (needs a leiden_r<RESOLUTION> column)
COLOR_BY = []                  # for example ["region", "sample", "centroid-0"]
GROUP_COL = None               # column that defines the groups in sections 5 and 6 (None = automatic)
FIGURE_SIZE = "full"           # "half", "full", "a4_landscape" or "a4_portrait"
COLOR_TRANSFORM = "raw"        # for numeric COLOR_BY columns: "raw", "rank", "log1p", "sqrt", "zscore" or "winsorize"

LABELS_TIFF = None             # optional: labels volume of ONE stack, to draw typical cells
LABELS_STACK_NAME = None       # the stack_name that LABELS_TIFF belongs to (None if the table has one stack)
VOXEL_SIZE_ZYX = (1.0, 1.0, 1.0)   # size of one voxel (z, y, x), in the units you want on the drawings
MESHES_DIR = None              # optional: the plugin's mesh folder, <MESHES_DIR>/<stack_name>/cell_<label>.obj
N_GALLERY_CELLS = 5            # typical cells shown per cluster

# ---------------- Settings with defaults ----------------
POINT_SIZE = 3                 # size of one cell in the map
POINT_ALPHA = 0.7              # 1 = solid, lower = see-through
SAVE_FIGURES = True

## 2. Load the table

The notebook reads the embedding, adds your optional per-stack columns, and
checks that the map and the cluster column are there.

In [ ]:
out_dir = Path(OUTPUT_DIR)
fig_dir = out_dir / "figures"
tab_dir = out_dir / "tables"
out_dir.mkdir(parents=True, exist_ok=True)
tab_dir.mkdir(exist_ok=True)

df = h.read_table_csv(SHAPE_EMBEDDING_CSV)
df = h.attach_metadata(df, METADATA_CSV)
for need in ("stack_name", "label", "umap_1", "umap_2"):
    if need not in df.columns:
        raise ValueError(f"The embedding file has no '{need}' column. Use the file written by the first notebook.")

cluster_col = h.res_name(RESOLUTION)
if cluster_col not in df.columns:
    have = [c for c in df.columns if c.startswith("leiden_r")]
    raise ValueError(f"No column '{cluster_col}'. The clusters in this file are: {have}. Change RESOLUTION.")

stacks = sorted(df["stack_name"].unique())
pc_cols = sorted([c for c in df.columns if re.fullmatch(r"PC\d+", c)], key=lambda c: int(c[2:]))
cluster_palette = h.assign_palette(df[cluster_col])
print(f"{len(df):,} cells, {len(stacks)} stack(s), {df[cluster_col].nunique()} clusters at resolution {RESOLUTION}.")

if GROUP_COL is None:
    GROUP_COL = "stack_name" if len(stacks) > 1 else next((c for c in COLOR_BY if c in df.columns and h.column_kind(df[c]) == "categorical"), None)
print("Group column for sections 5 and 6:", GROUP_COL)


def safe(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name))


def finish(fig, name):
    """Save the figure (SVG + PNG) and show it."""
    fig.tight_layout()
    if SAVE_FIGURES:
        h.save_fig(fig, name, fig_dir)
    plt.show()

## 3. The map coloured by shape cluster

Each dot is a cell. Cells close together have similar shapes. The legend gives
the number of cells in every cluster. The colours are the same as in the first
notebook.

In [ ]:
fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
h.umap_scatter(df, "umap_1", "umap_2", cluster_col, ax=ax, kind="categorical", palette=cluster_palette,
               s=POINT_SIZE, alpha=POINT_ALPHA, title=f"Shape clusters (resolution {RESOLUTION})", legend_counts=True)
finish(fig, "umap_clusters")

## 4. The same map, coloured by your columns

One panel for each column in `COLOR_BY`. A column with a handful of values
(text, or numbers such as 1, 2, 3) gets one colour per value. A measurement gets
a colour bar. Cells with an empty value are drawn in light grey.

For a skewed measurement, `COLOR_TRANSFORM` can help: `"rank"` spreads the colours
evenly over the cells, `"log1p"` compresses large values, `"zscore"` centres the scale.

In [ ]:
if not COLOR_BY:
    print("COLOR_BY is empty: add column names in the EDIT cell to get these panels.")
for col in COLOR_BY:
    if col not in df.columns:
        print(f"Column '{col}' is not in the table or the metadata; skipped.")
        continue
    kind = "categorical" if h.column_kind(df[col]) == "categorical" else "continuous"
    fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
    h.umap_scatter(df, "umap_1", "umap_2", col, ax=ax, kind=kind,
                   palette=h.assign_palette(df[col]) if kind == "categorical" else None,
                   transform=COLOR_TRANSFORM, s=POINT_SIZE, alpha=POINT_ALPHA, legend_counts=kind == "categorical")
    finish(fig, f"umap_by_{safe(col)}")

## 5. Which clusters make up each group?

For every group (a value of `GROUP_COL`), the bar shows how its cells divide
among the shape clusters. All bars add up to 1, so groups of different sizes can
be compared. The table behind the bars is saved to `tables/`.

In [ ]:
if GROUP_COL is None or GROUP_COL not in df.columns:
    print("No group column available: set GROUP_COL (or COLOR_BY) in the EDIT cell to get the bar and the heatmaps.")
else:
    comp = h.composition_table(df, GROUP_COL, cluster_col)
    comp.to_csv(tab_dir / f"cluster_share_by_{safe(GROUP_COL)}.csv")
    fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
    h.composition_bar(df, GROUP_COL, cluster_col, ax=ax, palette=cluster_palette, horizontal=comp.shape[0] > 8)
    ax.set_title(f"Cluster make-up of each {GROUP_COL}", fontsize=h.FONT_TITLE)
    finish(fig, f"cluster_share_by_{safe(GROUP_COL)}")

## 6. Group by cluster heatmaps

Left: the share of each group's cells that sit in each cluster (the numbers
behind the bars). Right: **over- or under-representation**. Each value is
log2(observed / expected): 0 = as often as chance, +1 = twice as often as
chance, -1 = half as often. "Expected" assumes the group does not matter. A grey square marked "none" is a pair that never occurs (no cell at all).

In [ ]:
if GROUP_COL is not None and GROUP_COL in df.columns:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(h.figure_size(FIGURE_SIZE)[0] * 1.7, h.figure_size(FIGURE_SIZE)[1]))
    comp = h.composition_table(df, GROUP_COL, cluster_col)
    h.matrix_heatmap(comp, ax=a1, cbar_label="share of the group's cells", cmap="Blues")
    a1.set_xlabel(cluster_col, fontsize=h.FONT_AXIS)
    a1.set_ylabel(GROUP_COL, fontsize=h.FONT_AXIS)
    _, enrich = h.enrichment_heatmap(df, GROUP_COL, cluster_col, ax=a2)
    enrich.to_csv(tab_dir / f"log2_enrichment_{safe(GROUP_COL)}_by_cluster.csv")
    pd.crosstab(df[GROUP_COL], df[cluster_col]).to_csv(tab_dir / f"cell_counts_{safe(GROUP_COL)}_by_cluster.csv")
    finish(fig, f"heatmaps_{safe(GROUP_COL)}_by_cluster")
else:
    print("No group column available (see section 5).")

## 7. One dot per stack

If your table holds several stacks (replicates), a result is only convincing when
it repeats. Here every dot is one stack: the share of that stack's cells in the
cluster. A tight row of dots means the cluster is found in every stack at a
similar share. A stack in which the cluster does not occur shows a dot at 0. With one stack this section is skipped.

In [ ]:
if len(stacks) > 1:
    fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
    ax, rep_tab = h.replicate_dotplot(df, cluster_col, ax=ax, palette=cluster_palette)
    rep_tab.to_csv(tab_dir / "cluster_share_per_stack.csv", index=False)
    finish(fig, "cluster_share_per_stack")
else:
    print("One stack in the table: the replicate plot is skipped.")

## 8. Numbered badges at the centre of each cluster

The cells are drawn faintly in their cluster colour. A numbered badge marks the
median position of every cluster, so a reader can find "cluster 4" quickly.

In [ ]:
fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
h.umap_with_medians(df, "umap_1", "umap_2", cluster_col, palette=cluster_palette, ax=ax, s=POINT_SIZE,
                    title=f"Shape clusters (resolution {RESOLUTION})")
finish(fig, "umap_cluster_badges")

## 9. A grey "ghost" map to finish in a vector editor

All cells are faint grey. Only the **most typical cell of each cluster** (the
one closest to the cluster's median, measured in PC space) is marked, with a
coloured diamond and a label. Open the SVG in Inkscape or Illustrator and place a
picture of the cell next to each marker. Every marker and label has its own
name in the SVG (`marker_<cluster>`, `label_<cluster>`). The table
`typical_cells.csv` lists the stack and label of each typical cell and its
position on the map. The typical cell is closest to the cluster's median in PC
space, so its marker can sit away from the badge of section 8, which marks the
median position on the map itself.

In [ ]:
if len(pc_cols) >= 2:
    typical = h.median_cells(df, pc_cols, cluster_col)
    typical.to_csv(tab_dir / "typical_cells.csv", index=False)
    fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
    h.umap_ghost(df, "umap_1", "umap_2", cluster_col, typical, palette=cluster_palette, ax=ax, prefix="")
    finish(fig, "umap_ghost")
    print(typical.round(3).to_string(index=False))
else:
    print("The file has no PC columns, so the typical cells cannot be found.")

## 10. Pictures of typical cells (optional)

A cluster is easier to understand when you look at its cells. For each cluster
the notebook takes the cells closest to the cluster's median in PC space and
draws them at the same scale.

- Give `LABELS_TIFF` (the labels volume of one stack) and the cells of that stack are drawn from it.
- Or give `MESHES_DIR` (the plugin's mesh folder) and the saved meshes are used.

Both are optional: without them this section is skipped.

In [ ]:
if (LABELS_TIFF is None and MESHES_DIR is None) or len(pc_cols) < 2:
    print("No LABELS_TIFF or MESHES_DIR given: pictures skipped. typical_cells.csv (section 9) lists the cells to look at.")
else:
    reps = h.select_representatives(df[pc_cols].to_numpy(dtype=float), df, cluster_col, N_GALLERY_CELLS)
    labels_volume = h.read_labels_volume(LABELS_TIFF) if LABELS_TIFF else None
    labels_stack = LABELS_STACK_NAME or (stacks[0] if len(stacks) == 1 else None)
    fig = h.plot_cluster_gallery(reps, cluster_palette, labels_volume, labels_stack, VOXEL_SIZE_ZYX, MESHES_DIR,
                                 n_per_cluster=N_GALLERY_CELLS)
    if fig is None:
        print("No mesh could be made for the typical cells (check the stack name and the folder).")
    else:
        finish(fig, "typical_cells_gallery")

## 11. What was written

All figures are in `figures/` as `.svg` (editable) and `.png`. The tables behind
them are in `tables/`.

In [ ]:
for sub in (fig_dir, tab_dir):
    names = sorted(p.name for p in sub.glob("*")) if sub.exists() else []
    print(f"{sub.name}/ ({len(names)} files):")
    for n in names:
        print("   ", n)